# صَوْن | Sawn

PDPL data-subject request agent for **Nakhla Retail**, a fictional online retailer (all data synthetic).
Capstone, Track 2 (Your harness): SDAIA Academy, Agentic AI Systems Engineering.

Runs on Colab with an OpenRouter key in the 🔑 Secrets panel (`OPENROUTER_API_KEY`, notebook access on).
Cells 1 to 5 use no model requests. Model requests per cell: 6 ≈ 3, 7 ≈ 50, 8 ≈ 50, 10 (live) ≈ 8.
The free tier allows about 50 a day, so run cell 7 one day and cell 8 the next; both resume where they stopped.

## 1 · Environment

Clones the course repository and installs its requirements. Sawn reuses three course modules: `agentkit` (agent loop,
MCP bridge, model client), `evals` (graders) and `madar` (`ToolError`).

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic-ai").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/SDAIAAcademy/agentic-ai.git", "/content/agentic-ai"], check=True)
    os.chdir("/content/agentic-ai/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("agentkit.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")
sys.path.insert(0, os.getcwd())

import agentkit as K
LABS = K.LABS
SAWN = LABS / "sawn"
print("labs:", LABS, "| provider:", K.PROVIDER, "| model:", K.MODEL)

## 2 · Project files

From GitHub when `SAWN_GITHUB` is set (a public repository), otherwise from an uploaded `sawn.zip`.

In [ ]:
SAWN_GITHUB = ""   # e.g. "https://github.com/MeshaalAlharthy/sawn"; leave empty to upload sawn.zip

import io, zipfile
if SAWN_GITHUB:
    if (SAWN / ".git").exists():
        subprocess.run(["git", "-C", str(SAWN), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", SAWN_GITHUB, str(SAWN)], check=True)
elif IN_COLAB:
    from google.colab import files
    for name, content in files.upload().items():
        zipfile.ZipFile(io.BytesIO(content)).extractall(LABS)
        print("extracted", name)

sys.path.insert(0, str(SAWN))
print(sorted(p.name for p in SAWN.iterdir() if not p.name.startswith((".", "__"))))

## 3 · Keep the cache and results across sessions

Links `labs/cache` (every model response) and `sawn/results` (eval CSVs, report, traces) to Google Drive, in
`MyDrive/sawn_capstone`. A Colab runtime is wiped when it disconnects; with this link an eval stopped by the daily
limit resumes the next day, and finished model calls replay from the cache at no cost. Set `USE_DRIVE = False` to skip.

In [ ]:
USE_DRIVE = True

def keep_on_drive(local: Path, target: Path) -> None:
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        return
    if local.exists():
        for f in local.rglob("*"):
            dest = target / f.relative_to(local)
            if f.is_file() and not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dest)
        shutil.rmtree(local)
    local.symlink_to(target, target_is_directory=True)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    keep = Path("/content/drive/MyDrive/sawn_capstone")
    keep_on_drive(LABS / "cache", keep / "cache")
    keep_on_drive(SAWN / "results", keep / "results")
    print("cache and results are kept in", keep)

## 4 · API key and model

Confirms the key is loaded; sends no request. To try another free tool-calling model, set `AGENT_MODEL` before
cell 1 (`os.environ["AGENT_MODEL"] = "..."`) and restart the runtime.

In [ ]:
try:
    key = K.api_key()
    print("✅ OPENROUTER_API_KEY found (" + key[:8] + "…) · model", K.MODEL, "· fallbacks", K.FALLBACKS)
except RuntimeError as e:
    print("❌", e)

## 5 · Offline tests

40 tests with a scripted stand-in model and the MCP server in-process: tool rules, retrieval, permissions, approvals,
guards, budgets, delivery, grader checks against reference trajectories, the eval runner, the HTTP service, the
OpenAI-compatible wire path (against a local stand-in server) and the attack. No model requests.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "sawn/tests"], capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

## 6 · One request with the real model

Runs one request from the inbox on a private copy of the database. Each write, and the outgoing reply, stops for your
decision in the input box below the cell (`y` approves). About 3 model requests.

In [ ]:
import harness as H
from sawn import Sawn

REQUEST_ID = "REQ-001"   # REQ-001 … REQ-010; see data/inbox.jsonl

inbox = {r["id"]: r for r in map(json.loads, (SAWN / "data" / "inbox.jsonl").read_text(encoding="utf-8").splitlines())}
item = inbox[REQUEST_ID]
db = Sawn(K.lab_dir("sawn/notebook"))
db.reset()
with H.policy_server("stdio") as mcp:
    o = H.handle(item["body"], sawn=db, tools=mcp.agent_tools(), approve=H.cli_approver(db), channel=item["channel"],
                 sender=item["from_addr"], on_event=H.print_event, raise_errors=False)
print("\n" + o.final)
print("\nreply:", o.delivery, "| model requests:", o.requests, "| tokens:", o.tokens)

## 7 · Eval, configuration A (baseline prompt)

17 tasks, about 50 model requests. Each finished task is saved at once; if the free tier's daily limit stops the run,
run this cell again the next day. `--limit 5` develops on a few tasks first.

In [ ]:
!cd {LABS} && python -u sawn/sawn_agent.py eval --config A

## 8 · Eval, configuration B (AGENTS.md spec)

The same 17 tasks with the spec as the system prompt; nothing else changes.

In [ ]:
!cd {LABS} && python -u sawn/sawn_agent.py eval --config B

## 9 · Report and error analysis

Writes `results/REPORT.md` and refreshes the evaluation section of `README.md`. No model requests.

In [ ]:
from IPython.display import Markdown
r = subprocess.run([sys.executable, "sawn/sawn_agent.py", "report"], capture_output=True, text=True, cwd=LABS)
report = SAWN / "results" / "REPORT.md"
if r.returncode == 0 and report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print(r.stderr[-1500:] or r.stdout[-1500:])

## 10 · The attack: stored prompt injection

Offline (no requests): a scripted compromised model obeys the instruction hidden in support ticket TCK-4008, against
the naive setup and the defended one. `LIVE = True` runs the same request through the real model (about 8 requests).

In [ ]:
LIVE = False
for args in (["attack"], ["attack", "--live"]) if LIVE else (["attack"],):
    r = subprocess.run([sys.executable, "-u", "sawn/sawn_agent.py", *args], capture_output=True, text=True, cwd=LABS)
    print(r.stdout[-3000:] or r.stderr[-3000:])

## 11 · Web console

Starts the HTTP service in the background and opens the officer's console. API key: `officer-dev-key`.
`OFFLINE = True` uses the scripted model for the sample requests (no model requests).

In [ ]:
import sawn_api

OFFLINE = False
PORT = 8000
server = sawn_api.serve_in_background(port=PORT, offline=OFFLINE)
if IN_COLAB:
    from google.colab import output
    print("Console in a new tab:", output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    output.serve_kernel_port_as_iframe(PORT, height=860)
else:
    print(f"Console: http://127.0.0.1:{PORT}")

## 12 · Download the project with its results

`sawn_final.zip` holds the whole project folder with `results/` and the refreshed `README.md`, ready for GitHub.

In [ ]:
import tempfile
staging = Path(tempfile.mkdtemp()) / "sawn"
shutil.copytree(SAWN, staging, symlinks=False, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", "logs", ".git"))
archive = shutil.make_archive("/content/sawn_final" if IN_COLAB else "sawn_final", "zip", staging.parent, "sawn")
print(archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)